In [ ]:
# Install Detectron2
import sys
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
# Clone the RSKT-Seg Architecture
!git clone https://github.com/LiBingyu01/RSKT-Seg_and_Pi-Seg.git

In [ ]:
# Install requirements
%cd /kaggle/working/RSKT-Seg_and_Pi-Seg
!sed -i 's/==/>=/g' requirements.txt
!pip install -r requirements.txt

In [ ]:
%%bash
cd /kaggle/working/RSKT-Seg_and_Pi-Seg

# Weights Setup
mkdir -p pretrained
ln -sf /kaggle/input/datasets/<username>/rskt-potsdam-test-data/rskt-weights/*.pt ./pretrained/
ln -sf /kaggle/input/datasets/<username>/rskt-potsdam-test-data/rskt-weights/*.pth ./pretrained/
# Explicitly link model_final.pth so the script finds it
ln -sf /kaggle/input/datasets/<username>/rskt-potsdam-test-data/rskt-weights/model_final.pth ./pretrained/rskt_seg_vit_l.pth

# Bypass PyTorch 2.6 security block
sed -i "s/torch.load(Weights, map_location='cpu')/torch.load(Weights, map_location='cpu', weights_only=False)/g" RSKT_Seg/RSKT_Seg.py

# Folder Setup & Python Registry Patch
# The RSKT-Seg model has these exact folder paths hardcoded inside its source code.
mkdir -p datasets/Potsdam/imgs
mkdir -p datasets/Potsdam/D2masks
# register_Potsdam.py looks for .png files but our downloaded dataset is .tif
sed -i 's/gt_ext="png", image_ext="png"/gt_ext="tif", image_ext="tif"/g' RSKT_Seg/data/datasets/register_Potsdam.py

# Link Images (stripping _RGB so Detectron2 can pair them)
SOURCE="/kaggle/input/datasets/<username>/rskt-potsdam-test-data/rskt-weights"
for img in "$SOURCE"/*_RGB.tif; do
    filename=$(basename "$img")
    base="${filename%_RGB.tif}"
    ln -sf "$img" "./datasets/Potsdam/imgs/${base}.tif"
done

In [ ]:
# The original images have 3 channel (RGB), but the model can only work on 1 channel, i.e, a 2D matrix
import os
import numpy as np
from PIL import Image

source_dir = "/kaggle/input/datasets/<username>/rskt-potsdam-test-data/rskt-weights"
dest_dir = "/kaggle/working/RSKT-Seg_and_Pi-Seg/datasets/Potsdam/D2masks"

# Mapped to strictly start at 0, with Clutter set to 255 (Ignore)
color2id = {
    (255, 255, 255): 0, # Impervious surfaces
    (0, 0, 255): 1,     # Building
    (0, 255, 255): 2,   # Low vegetation
    (0, 255, 0): 3,     # Tree
    (255, 255, 0): 4,   # Car
    (255, 0, 0): 255,   # Clutter/Background (IGNORE)
}

print("Converting original RGB masks directly to Contiguous Index masks...")

for filename in os.listdir(source_dir):
    if filename.endswith("_label_noBoundary.tif"):
        # Get the base name (e.g., top_potsdam_5_14.tif)
        base_name = filename.replace("_label_noBoundary.tif", ".tif")

        source_path = os.path.join(source_dir, filename)
        dest_path = os.path.join(dest_dir, base_name)

        # Open the original raw RGB image from the read-only dataset
        img = Image.open(source_path).convert('RGB')
        arr = np.array(img)

        mask = np.full((arr.shape[0], arr.shape[1]), 255, dtype=np.uint8)

        for color, class_id in color2id.items():
            matches = (arr == color).all(axis=-1)
            mask[matches] = class_id

        # Save directly over the old files in the working directory
        Image.fromarray(mask).save(dest_path)
        print(f"Successfully converted and saved: {base_name}")

In [ ]:
import os
import sys
import cv2
import torch
import math
import numpy as np
from detectron2.data import DatasetCatalog
from detectron2.engine import default_argument_parser
import detectron2.data.transforms as T
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from PIL import Image

base_dir = "/kaggle/working/RSKT-Seg_and_Pi-Seg/"
os.chdir(base_dir)
sys.path.insert(0, base_dir)
import train_net

print("Building Configuration...")
args = default_argument_parser().parse_args([
    "--config-file", "configs/vitl_336_DLRSD.yaml",
    "--eval-only",
    "MODEL.WEIGHTS", "pretrained/rskt_seg_vit_l.pth"
])
cfg = train_net.setup(args)
cfg.defrost()
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Weights...")
model = build_model(cfg)
model.eval()
DetectionCheckpointer(model).load(cfg.MODEL.WEIGHTS)
aug = T.ResizeShortestEdge([cfg.INPUT.MIN_SIZE_TEST, cfg.INPUT.MIN_SIZE_TEST], cfg.INPUT.MAX_SIZE_TEST)

dataset_dicts = DatasetCatalog.get("Potsdam_all_sem_seg")
test_dataset = dataset_dicts[:6]

# Create the folder to save predictions
output_dir = base_dir + "datasets/Potsdam/predictions/"
os.makedirs(output_dir, exist_ok=True)

# Define the official Potsdam color mapping
id2color = {
    0: [255, 255, 255], # Impervious surfaces (White)
    1: [0, 0, 255],     # Building (Blue)
    2: [0, 255, 255],   # Low vegetation (Cyan)
    3: [0, 255, 0],     # Tree (Green)
    4: [255, 255, 0],   # Car (Yellow)
    5: [255, 0, 0],     # Clutter/Ignore (Red) - Fallback
    6: [255, 0, 0]      # Clutter/Ignore (Red)
}

matrix_size = 6
global_conf_matrix = np.zeros((matrix_size, matrix_size), dtype=np.int64)
translation_map = torch.tensor([6, 6, 1, 4, 2, 0, 0, 2, 2, 1, 0, 6, 6, 6, 6, 3, 6], device=cfg.MODEL.DEVICE)

PATCH_SIZE = 512
STRIDE = 256

print(f"\nStarting Sliding Window Evaluation & Caching on {len(test_dataset)} images...")

with torch.no_grad():
    for idx, d in enumerate(test_dataset):
        file_basename = os.path.basename(d['file_name'])
        print(f"Processing & Saving {idx + 1}/{len(test_dataset)}: {file_basename}")

        im = cv2.imread(d["file_name"])
        gt_mask = cv2.imread(d["sem_seg_file_name"], cv2.IMREAD_GRAYSCALE)
        full_h, full_w = im.shape[:2]

        accumulated_logits = torch.zeros((17, full_h, full_w), dtype=torch.float32, device="cpu")
        count_map = torch.zeros((1, full_h, full_w), dtype=torch.float32, device="cpu")

        # Slicing
        for y in range(0, full_h, STRIDE):
            for x in range(0, full_w, STRIDE):
                y1, y2 = y, min(y + PATCH_SIZE, full_h)
                x1, x2 = x, min(x + PATCH_SIZE, full_w)
                if y2 - y1 < PATCH_SIZE: y1 = max(0, y2 - PATCH_SIZE)
                if x2 - x1 < PATCH_SIZE: x1 = max(0, x2 - PATCH_SIZE)

                patch_im = im[y1:y2, x1:x2]
                patch_tensor = aug.get_transform(patch_im).apply_image(patch_im)
                patch_tensor = torch.as_tensor(patch_tensor.astype("float32").transpose(2, 0, 1)).to(cfg.MODEL.DEVICE)

                inputs = {"image": patch_tensor, "height": PATCH_SIZE, "width": PATCH_SIZE, "file_name": d["file_name"]}

                outputs = model([inputs])[0]
                accumulated_logits[:, y1:y2, x1:x2] += outputs["sem_seg"].cpu()
                count_map[:, y1:y2, x1:x2] += 1

        raw_prediction = (accumulated_logits / count_map).argmax(dim=0).to(cfg.MODEL.DEVICE)
        mapped_prediction = translation_map[raw_prediction].cpu().numpy()

        rgb_prediction = np.zeros((full_h, full_w, 3), dtype=np.uint8)
        for class_id, color in id2color.items():
            rgb_prediction[mapped_prediction == class_id] = color

        # Save as a standard colored PNG instead of a TIF
        save_name = file_basename.replace(".tif", ".png")
        save_path = os.path.join(output_dir, save_name)
        Image.fromarray(rgb_prediction).save(save_path)

        # The metric logic below remains untouched and uses the original mapped_prediction!
        pred_flat = mapped_prediction.flatten()
        gt_flat = gt_mask.flatten()

        valid_pixels = (gt_flat != 255) & (gt_flat != 6)
        gt_valid = gt_flat[valid_pixels]
        pred_valid = pred_flat[valid_pixels]

        pred_valid[pred_valid >= 5] = 5

        img_conf_matrix = np.bincount(
            matrix_size * gt_valid + pred_valid,
            minlength=matrix_size**2
        ).reshape(matrix_size, matrix_size)

        global_conf_matrix += img_conf_matrix

print("\n" + "="*50)
print("--- CORE 5x5 CONFUSION MATRIX (Ignoring Clutter) ---")
print(global_conf_matrix[:5, :5])

intersection = np.diag(global_conf_matrix)[:5]
ground_truth_set = global_conf_matrix[:5, :].sum(axis=1)
predicted_set = global_conf_matrix[:, :5].sum(axis=0)

gt_safe = np.maximum(ground_truth_set, 1)
pred_safe = np.maximum(predicted_set, 1)
union = ground_truth_set + predicted_set - intersection
union_safe = np.maximum(union, 1)

# Overall Pixel Accuracy
total_correct = intersection.sum()
total_pixels = ground_truth_set.sum()
accuracy = (total_correct / max(total_pixels, 1)) * 100

# Recall & Precision
recall = (intersection / gt_safe) * 100
precision = (intersection / pred_safe) * 100

# F1-Score & IoU
iou = (intersection / union_safe) * 100
f1_scores = 2 * (precision * recall) / (precision + recall)
f1_scores = np.nan_to_num(f1_scores)

print(f"\n--- OVERALL PIXEL ACCURACY ---")
print(f"{accuracy:.2f}%")

print("\n--- GLOBAL PER-CLASS RECALL (%) ---")
print(np.round(recall, 2))
print(f"Mean Recall: {np.mean(recall):.2f}%")

print("\n--- GLOBAL PER-CLASS IoU (%) ---")
print(np.round(iou, 2))
print(f"Mean IoU: {np.mean(iou):.2f}%")

print("\n--- GLOBAL PER-CLASS F1-SCORE (%) ---")
print(np.round(f1_scores, 2))
print(f"Mean F1: {np.mean(f1_scores):.2f}%")
print("="*50 + "\n")